In [ ]:
# Install ipython-sql via pip
!pip install duckdb numpy pandas

In [12]:
import duckdb

con = duckdb.connect(database=':memory:')

query = """WITH SplitCTE AS (
    SELECT
        *,
        STRING_SPLIT(tx, ' ') AS tx_parts,
        STRING_SPLIT(rx, ' ') AS rx_parts
    FROM read_ndjson_auto('./data/**/*.ndjson')
    WHERE tx LIKE '00 __ 00 %'
    AND rx NOT LIKE '00 __ 00 %'
    AND rx NOT LIKE '00 __ 02 __ 01 %'
)
SELECT src, dev, tx, rx, tx_parts, rx_parts FROM SplitCTE;"""

# Execute SQL
con.execute(query)

# Optionally fetch results from the last query
result = con.fetchdf()
result


,src,dev,tx,rx,tx_parts,rx_parts
0,mvg,BT-E6000,00 1A 00 A6 67,00 9A 07 A6 A1 17 60 56 52 4B F2 9F,"[00, 1A, 00, A6, 67]","[00, 9A, 07, A6, A1, 17, 60, 56, 52, 4B, F2, 9F]"
1,mvg,BT-E6000,00 92 00 AA 25,00 92 03 AA 90 01 5E B6,"[00, 92, 00, AA, 25]","[00, 92, 03, AA, 90, 01, 5E, B6]"
2,mvg,BT-E6000,00 97 00 12 5B,00 97 0A 12 25 00 E1 00 00 90 10 0F 00 34 14,"[00, 97, 00, 12, 5B]","[00, 97, 0A, 12, 25, 00, E1, 00, 00, 90, 10, 0..."
3,mvg,BT-E6000,00 A9 00 A0 77,00 A9 2B A0 00 00 00 00 1F 00 00 00 34 00 34 0...,"[00, A9, 00, A0, 77]","[00, A9, 2B, A0, 00, 00, 00, 00, 1F, 00, 00, 0..."
4,mvg,BT-E6000,00 B7 00 21 78,00 B7 03 21 25 FF F6 8F,"[00, B7, 00, 21, 78]","[00, B7, 03, 21, 25, FF, F6, 8F]"
5,mvg,BT-E6000,00 B9 00 31 E2,00 B9 12 31 25 9F 01 A9 01 01 00 05 00 28 00 6...,"[00, B9, 00, 31, E2]","[00, B9, 12, 31, 25, 9F, 01, A9, 01, 01, 00, 0..."


It seems like this exposes some possible commands, I would like to run a test where the first CRC byte increments by one.  
I suspect it will expose all commands the device knows.  
  
CRC byte was a bit unpredictable, so I created short messages for each possible command/type byte.  

In [ ]:
# Possible Commands
import duckdb

con = duckdb.connect(database=':memory:')

query = """WITH SplitCTE AS (
    SELECT
        *,
        STRING_SPLIT(tx, ' ') AS tx_parts,
        STRING_SPLIT(rx, ' ') AS rx_parts
    FROM read_ndjson_auto('./data/**/*.ndjson')
    WHERE intent = 'Command sniffing'
    AND tx LIKE '00 __ 02 %'
    AND (rx NOT LIKE '00 __ 02 __ 01 %' or rx = '')
    
)
SELECT dev, tx, rx, notes FROM SplitCTE;"""

# Execute SQL
con.execute(query)

# Optionally fetch results from the last query
result = con.fetchdf()
result

,dev,tx,rx,notes
0,BT-E6000,00 40 02 10 00 40 CA,00 C0 16 10 25 00 00 02 00 E5 00 66 00 08 00 1...,OK
1,BT-E6000,00 41 02 11 00 23 CF,00 C1 09 11 25 47 00 0F 03 00 00 64 A0 B9,OK
2,BT-E6000,00 42 02 12 00 86 C0,00 C2 0A 12 25 00 E1 00 00 90 10 0F 00 AF D6,OK
3,BT-E6000,00 43 02 13 00 E5 C5,00 C3 20 13 25 00 01 01 00 04 0A 01 02 52 0A 0...,OK
4,BT-E6000,00 40 02 20 00 E2 7C,00 C0 03 20 25 00 7D FF,OK
5,BT-E6000,00 41 02 21 00 81 79,00 C1 03 21 25 FF 9D A1,OK
6,BT-E6000,00 40 02 30 00 73 E9,00 C0 02 30 12 8E F7,OK
7,BT-E6000,00 41 02 31 00 10 EC,,TIMEOUT
8,BT-E6000,00 42 02 32 00 B5 E3,00 C2 02 32 12 48 FD,OK
9,BT-E6000,00 40 02 A0 00 2E F0,00 C0 2B A0 00 00 00 00 21 00 0E 00 33 00 32 0...,OK


Querying the data I noticed that for sender ID 0xB9 command 0x31 does return data. 

In [11]:
import duckdb

con = duckdb.connect(database=':memory:')

query = """WITH SplitCTE AS (
    SELECT
        *,
        STRING_SPLIT(tx, ' ') AS tx_parts,
        STRING_SPLIT(rx, ' ') AS rx_parts
    FROM read_ndjson_auto('./data/**/*.ndjson')
    WHERE intent LIKE 'Command sniffing%'
    AND tx LIKE '00 B9 01 %'
    AND (rx NOT LIKE '00 __ 02 __ 01 %' or rx = '')
    
)
SELECT dev, tx, rx, notes FROM SplitCTE;"""

# Execute SQL
con.execute(query)

# Optionally fetch results from the last query
result = con.fetchdf()
result

,dev,tx,rx,notes
0,BT-E6000,00 B9 01 10 C9 D9,00 B9 01 10 C9 D9,OK
1,BT-E6000,00 B9 01 11 40 C8,00 B9 09 11 25 47 00 0F 03 00 00 64 C0 5B,OK
2,BT-E6000,00 B9 01 12 DB FA,00 B9 0A 12 25 00 E1 00 00 90 10 0F 00 A5 32,OK
3,BT-E6000,00 B9 01 13 52 EB,00 B9 02 13 25 CF 18,OK
4,BT-E6000,00 B9 01 20 4A E8,00 B9 03 20 25 00 EA BB,OK
5,BT-E6000,00 B9 01 21 C3 F9,00 B9 03 21 25 FF 4E EE,OK
6,BT-E6000,00 B9 01 30 CB F8,00 B9 02 30 12 A8 54,OK
7,BT-E6000,00 B9 01 31 42 E9,00 B9 12 31 25 9F 01 A9 01 01 00 05 00 28 00 6...,OK
8,BT-E6000,00 B9 01 32 D9 DB,00 B9 02 32 12 18 67,OK
9,BT-E6000,00 B9 01 A0 42 6C,00 B9 2B A0 00 00 00 00 21 00 0B 00 34 00 34 0...,OK


0xB9 sender does return for 0x31 type, but no other types suddenly return.

In [5]:
# Unknown command/type reply?
import duckdb

con = duckdb.connect(database=':memory:')

query = """WITH SplitCTE AS (
    SELECT
        *,
        STRING_SPLIT(tx, ' ') AS tx_parts,
        STRING_SPLIT(rx, ' ') AS rx_parts
    FROM read_ndjson_auto('./data/**/*.ndjson')
    WHERE intent = 'Command sniffing'
    AND NOT (tx LIKE '00 __ 02 %'
    AND (rx NOT LIKE '00 __ 02 __ 01 %' or rx = ''))
    
)
SELECT dev, tx, rx, notes FROM SplitCTE;"""

# Execute SQL
con.execute(query)

# Optionally fetch results from the last query
result = con.fetchdf()
result

,dev,tx,rx,notes
0,BT-E6000,00 40 05 00 00 00 00 00 F1 5F,00 C0 02 00 01 36 63,OK
1,BT-E6000,00 40 02 00 00 D1 5F,00 C0 02 00 01 36 63,OK
2,BT-E6000,00 41 02 01 00 B2 5A,00 C1 02 01 01 55 66,OK
3,BT-E6000,00 42 02 02 00 17 55,00 C2 02 02 01 F0 69,OK
4,BT-E6000,00 43 02 03 00 74 50,00 C3 02 03 01 93 6C,OK
...,...,...,...,...
294,BT-E6000,00 40 05 50 00 00 00 00 93 2A,00 C0 02 50 01 C1 B0,OK
295,BT-E6000,00 41 05 51 00 00 00 00 02 BE,00 C1 02 51 01 A2 B5,OK
296,BT-E6000,00 42 05 52 00 00 00 00 A0 0B,00 C2 02 52 01 07 BA,OK
297,BT-E6000,00 43 05 53 00 00 00 00 31 9F,00 C3 02 53 01 64 BF,OK
